# Brain Tumor MRI Classification (Multi-class)

This notebook trains a convolutional neural network to classify brain MRI images into four tumor-related classes using a custom CNN built with TensorFlow/Keras.

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import tensorflow as tf

2025-12-18 08:20:39.358445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766046039.539201      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766046039.591847      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

## Load and preprocess data

Define image size and batch size, then load the MRI images from the dataset directory as TensorFlow datasets with labels.


In [4]:
img_size = (224, 224)
batch_size = 34

In [6]:
train_df = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/brain-tumor-mri-multi-class-dataset/multi_class_dataset',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True
)

Found 16269 files belonging to 4 classes.


I0000 00:00:1766046099.828442      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


### Train/validation/test split

Split the full dataset into training, validation, and test sets using a 20% test split and 15% validation split from the remaining data.


In [8]:
df_size  = tf.data.experimental.cardinality(train_df).numpy()
test_size = int(0.20 * df_size)      # 20% test
val_size  = int(0.15 * (df_size - test_size))  # 15% val from remaining

test_df  = train_df.take(test_size)
train_df  = train_df.skip(test_size)
val_df   = train_df.take(val_size)
train_df = train_df.skip(val_size)

## Model architecture

Build a custom CNN with multiple convolution–batch normalization–max pooling blocks followed by dense layers and dropout for regularization.


In [14]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),

    # Block 1
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Block 2
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Block 3
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Block 4 (optional but useful)
    tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(4, activation="softmax")
])

### Compile model

Compile the model with Adam optimizer, sparse categorical cross-entropy loss, and accuracy as the evaluation metric.
|

In [15]:
model.compile(
    optimizer='adam',
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Early stopping callback

Use early stopping on validation loss to prevent overfitting and restore the best model weights.


In [17]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [18]:
history = model.fit(
    train_df,                
    validation_data=val_df,  
    epochs=50,
    callbacks=[early_stop]
)

Epoch 1/50


I0000 00:00:1766046243.811632     115 service.cc:148] XLA service 0x79bc040156b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766046243.812512     115 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1766046244.405891     115 cuda_dnn.cc:529] Loaded cuDNN version 90300


  5/327 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.3286 - loss: 2.1941 

I0000 00:00:1766046250.118963     115 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


327/327 ━━━━━━━━━━━━━━━━━━━━ 41s 76ms/step - accuracy: 0.6763 - loss: 0.9477 - val_accuracy: 0.8055 - val_loss: 0.5642
Epoch 2/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.8388 - loss: 0.4413 - val_accuracy: 0.5743 - val_loss: 1.1758
Epoch 3/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.8427 - loss: 0.4267 - val_accuracy: 0.8658 - val_loss: 0.3783
Epoch 4/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.8865 - loss: 0.3098 - val_accuracy: 0.8751 - val_loss: 0.3187
Epoch 5/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.9010 - loss: 0.2529 - val_accuracy: 0.8421 - val_loss: 0.4463
Epoch 6/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.9314 - loss: 0.1970 - val_accuracy: 0.8235 - val_loss: 0.4588
Epoch 7/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.9274 - loss: 0.2002 - val_accuracy: 0.9458 - val_loss: 0.1578
Epoch 8/50
327/327 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.9522 - loss: 0.1277 - val_accurac

## Evaluate on test set

Evaluate the final model on the held-out test set and report test loss and accuracy.


In [19]:
test_loss, test_acc = model.evaluate(test_df)
print("Test Loss: ", test_loss)
print("Test Acc: ", test_acc)

95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9663 - loss: 0.1070
Test Loss:  0.09539932757616043
Test Acc:  0.9681114554405212
